# Atelier Prompt Engineering — AI Business Assistant 
Ce notebook regroupe l'ensemble des réponses à l'atelier, organisées partie par partie.
## Contexte

L’entreprise souhaite mettre en place **AI Business Assistant**, un assistant IA polyvalent pour exploiter des documents, analyser des données, faire du machine learning et produire des résultats structurés. L’objectif de l’atelier est de construire progressivement et d’améliorer les prompts permettant de réaliser ces différentes tâches.

# Partie 1 — Anatomie d’un prompt

### Problème
Analyser les retours de clients d’une entreprise.

**Décomposition en composantes** : rôle, contexte, tâche, contraintes/format.


**Rôle**: Tu es un analyste spécialisé dans l’analyse des retours clients.

**Contexte** : Une entreprise de e‑commerce reçoit chaque semaine un grand volume d’avis clients provenant de différentes sources — réseaux sociaux, courriels et formulaires de satisfaction. Ces retours, bien que riches en informations, ne font pas encore l’objet d’une analyse systématique.

**Tâche**: Analyse l’ensemble des avis clients fournis ci‑dessous et produis une synthèse structurée mettant en évidence les tendances, les points forts et les axes d’amélioration.

**Contraintes**: 
1.Indique pour chaque avis le sentiment général : positif, négatif ou neutre.

2.Regroupe les avis par sujet : livraison, produit, service client, prix, application.

3.Mets en avant les points positifs les plus cités et les problèmes les plus fréquents.

4.Résume les tendances principales observées dans les retours.

5.Propose deux ou trois actions concrètes pour améliorer l’expérience client.

6.Utilise uniquement les informations présentes dans les avis, sans ajout ni interprétation.

7.Rédige de façon claire, neutre et concise, sans jugement personnel.

**Format de sortie** :
1. Points positifs
2. Problèmes principaux
3. Tendances
4. Recommandations

----
## Partie 2 — Comparer les techniques de prompting

**Objectif** : tester et comparer les résultats obtenus avec différentes approches de prompt (zero‑shot, one‑shot, few‑shot et prompt structuré).

**Tâche** : classer le commentaire suivant selon le sentiment exprimé :

#### <mark style= "background-color: lightblue">**Zero-shot** </mark>
📋**Prompt:**

Classe le commentaire suivant parmi : positif, négatif, neutre.
Commentaire : « Le service est rapide mais l'application plante régulièrement. »

💬**Réponse du LLM**: <mark> Neutre </mark>


**Evaluation**: Sans exemple ni règle de décision, le modèle traite les deux informations (rapidité / plantages) comme équivalentes en poids et choisit la catégorie "neutre" par défaut face à l'ambiguïté, faute d'indication sur comment trancher.



#### <mark style= "background-color: lightblue">**One-shot**</mark>
📋**Prompt:**

Exemple :

Commentaire : « Le livreur a été très aimable et ponctuel. »
Classe : positif

Maintenant, classe le commentaire suivant parmi positif, négatif ou neutre :
« Le service est rapide mais l'application plante régulièrement. »

💬**Réponse du LLM:**<mark>négatif</mark>

 **Evaluation:** L'exemple positif unique manque de nuance : il n'apprend pas au modèle à peser le pour et le contre d'un avis mixte, ce qui le pousse à privilégier la critique technique (« plante ») et à trancher arbitrairement vers le négatif.




#### <mark style= "background-color: lightblue">**Few-shot** </mark>
📋**Prompt** : 

Classe chaque texte en positif, négatif ou neutre.

Texte : « Le livreur a été très aimable et ponctuel. » → positif

Texte : « Ma commande n’est jamais arrivée. » → négatif

Texte : « Le produit correspond à la description, rien de plus. » → neutre

Texte : « Prix correct mais le service client ne répond jamais. » → négatif

Texte : « Le service est rapide mais l’application plante régulièrement. » → ?

💬**Réponse du LLM:** <mark>Neutre</mark>

**Evaluation:** Grâce aux multiples exemples, le modèle comprend mieux la tâche, mais il hésite toujours sur les avis partagés. Comme le prompt ne lui dit pas si un bug technique est plus grave qu'un service rapide, le modèle choisit neutre au hasard plutôt que négatif.

#### <mark style= "background-color: lightblue">**Prompt structuré** </mark>

📋**Prompt**

**Rôle**: Tu es un système d'analyse de sentiment client pour une entreprise de services numériques.

**Contexte**: Les commentaires clients peuvent contenir plusieurs aspects, parfois contradictoires (points positifs et négatifs dans le même commentaire).

**Tâche** : Classer le commentaire suivant dans une seule catégorie parmi : positif, négatif, neutre.

**Règles de décision** :
- Si le commentaire est globalement favorable sans défaut majeur → positif
- Si le commentaire mentionne un défaut qui impacte fortement l'usage (bug bloquant, panne, dysfonctionnement récurrent), même en présence d'un point positif → négatif
- Si les aspects positifs et négatifs sont équivalents en importance, sans dominance claire → neutre

**Format de sortie** : un seul mot parmi "positif", "négatif" ou "neutre", sans explication ni ponctuation.

**Commentaire à classer** : « Le service est rapide mais l'application plante régulièrement. »

--

💬**Réponse de LLM:** <mark>Négatif</mark>


**Évaluation (Prompt structuré)**

* **Points forts :** L'ajout de règles de décision explicites enlève toute ambiguïté sur la gestion des avis mixtes. Le modèle applique directement la règle stipulant qu'un dysfonctionnement récurrent ("plante régulièrement") implique une classification **négatif**.
* **Respect des contraintes :** Le format imposé (un seul mot, sans explication ni ponctuation) est parfaitement suivi, facilitant une exploitation automatique.
* **Bilan :** C'est la méthode la plus fiable et constante pour cadrer le comportement du modèle sur des tâches complexes.

## Partie 3 — Prompt Engineering et raisonnement

### 1. Décomposition du prompt
 « Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations » manque de structure. Voici son anatomie :
 
 -  Tâche principal (Instruction) : Analyser des avis clients, identifier des problèmes et suggérer des actions. 
 - Entrée (Données) : Absente (le texte d'avis clients à analyser n'est pas fourni). 
 - Rôle / Persona : Absent (pas -de rôle d'analyste expérience client spécifié).
 - Contraintes et Format : Absents (aucun format précisé comme JSON ou puces, pas de limite de mots, pas de définition de ce qu'est un problème "important").

### Prompt amélioré
📋**Prompt**

Rôle : Tu es un analyste chargé d'exploiter des avis clients pour une entreprise de e-commerce.

Tâche : À partir des avis fournis ci-dessous, identifie les 3 problèmes les plus fréquemment mentionnés et propose une recommandation actionnable pour chacun.

Contraintes :
- Base-toi uniquement sur le contenu des avis fournis, sans aucune exception.
- N'invente aucune information, cause, chiffre ou détail absent des avis.
- Ne suppose aucune intention ou émotion du client qui ne soit pas explicitement exprimée.
- Si moins de 3 problèmes distincts sont identifiables, n'en liste que le nombre réellement présent — ne complète pas artificiellement.
- Chaque problème cité doit être accompagné d'une référence à l'avis correspondant (numéro de l'avis).
- Chaque recommandation doit être directement liée au problème identifié, sans généralité vague (ex : interdiction de recommandations du type "améliorer la qualité de service" sans précision).
- N'utilise aucune connaissance externe sur l'entreprise, le secteur ou des cas similaires.

Format de sortie : Liste numérotée. Pour chaque élément : Problème (avec n° d'avis) → Recommandation.

Avis clients :
1. « Livraison en retard de 3 jours, aucune excuse reçue. »
2. « Le produit est de bonne qualité mais l'emballage était endommagé. »
3. « Service client injoignable pendant plus d'une semaine. »
4. « Très satisfait, rien à signaler. »
5. « Le prix a augmenté sans prévenir, c'est frustrant. »

💬**Réponse de LLM:**

1. Problème (Avis 1 & 3) : Retards de livraison sans communication et service client totalement injoignable pendant plus d'une semaine. → Recommandation : Rétablir un canal de communication fonctionnel pour répondre aux clients et envoyer une notification automatique d'excuse lors des retards de livraison.

2. Problème (Avis 2) : Emballages des colis arrivant endommagés lors de la réception. → Recommandation : Renforcer la protection physique des emballages lors du conditionnement avant l'expédition.

3. Problème (Avis 5) : Augmentation des prix appliquée sans notification préalable aux clients. → Recommandation : Informer à l'avance les clients de toute hausse de tarif avant son application.
